In [49]:
# Surface source + Floquet periodic BCs + WBM

from netgen.occ import *
from netgen.meshing import IdentificationType
from ngsolve import *
from ngsolve.webgui import Draw

# 1. Geometry Setup
L = 2.0
Lx = L/4
Ly = L/32

# Upper Air (Between Source and Rigid Wall)
geometry = Box((0, 0, 0), (Lx, Ly, L))
geometry.mat("air")     # <-- Added material name

# Tag the boundaries for Boundary Conditions
geometry.faces.Max(Z).name = "top"
geometry.faces.Min(Z).name = "bottom"

# 3. Identify Periodic Faces (on the outer boundaries of the glued geometry)
geometry.faces.Max(X).Identify(geometry.faces.Min(X), "periodic_x")
geometry.faces.Max(Y).Identify(geometry.faces.Min(Y), "periodic_y")

# 4. Generate Mesh
geo = OCCGeometry(geometry)
mesh = Mesh(geo.GenerateMesh(maxh=0.2))

print("Mesh generated successfully! Notice the internal face at z=0.2")
Draw(mesh)

Mesh generated successfully! Notice the internal face at z=0.2


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [51]:
import cmath, math
# 1. Physics Parameters
freq = 343.0     
c_0 = 343.0* (1 - 1j * 0.001)  # Complex speed of sound to introduce a small amount of damping
k = 2 * math.pi * freq / c_0  

# Incident angles (e.g., 45 degrees)
theta = (math.pi / 4)  # Polar angle (0 for normal incidence)
phi = 0.0

# Wave vector components
kx = k * math.sin(theta) * math.cos(phi)
ky = k * math.sin(theta) * math.sin(phi)
kz = k * math.cos(theta)

# Transverse wave vector for the Floquet shift
k_vec = CF((kx, ky, 0)) 

# 2. Finite Element Space
fes = Periodic(H1(mesh, order=3, complex=True))

u, v = fes.TnT()

print(f"Degrees of freedom: {fes.ndof}")
print(f"wave vector k: ({kx:.2f}, {ky:.2f}, {kz:.2f})")


Degrees of freedom: 1746
wave vector k: (4.44+0.00j, 0.00+0.00j, 4.44+0.00j)


In [52]:
# FEM model

# 2. Modified Gradients (Floquet Trick)
grad_u = grad(u) + 1j * k_vec * u
grad_v = grad(v) - 1j * k_vec * v 

# 3. Bilinear Form (LHS)
Z_fem = BilinearForm(fes)
Z_fem += (grad_u * grad_v - k**2 * u * v) * dx

# 4. Linear Form (RHS) - The Transparent Source
s_fem = LinearForm(fes)
source_func = 1.0  
s_fem += source_func* v * ds("bottom")

# 5. Assemble and Solve
gfu = GridFunction(fes)

with TaskManager():
    Z_fem.Assemble()
    s_fem.Assemble()
    gfu.vec.data = Z_fem.mat.Inverse(fes.FreeDofs()) * s_fem.vec
    
freedofs = fes.FreeDofs()
free_indices = [i for i, is_free in enumerate(freedofs) if is_free]
slave_indices = [i for i, is_free in enumerate(freedofs) if not is_free]
print("Solve complete!")


Solve complete!


In [53]:
# WBM model
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla

nWave = 0
m_indices = np.arange(-nWave, nWave + 1) # Harmonics in x: -nWave, ..., nWave
n_indices = np.arange(-nWave, nWave + 1) # Harmonics in y: -nWave, ..., nWave
total_waves = len(m_indices) * len(n_indices)

wave_kx = []
wave_ky = []
wave_kz = []
wave_functions = []

for m in m_indices:
    for n in n_indices:
        # Transverse Bloch-Floquet wavenumbers
        kx_wbm = 2 * np.pi * m / Lx
        ky_wbm = 2 * np.pi * n / Ly
        
        # Calculate axial wavenumber (kz), ensuring proper decay for evanescent waves
        val = k**2 - (kx + kx_wbm)**2 - (ky + ky_wbm)**2
        if val.real >= 0:
            kz_wbm = np.sqrt(val)
        else:
            kz_wbm = 1j * np.sqrt(-val) # Decay in +z direction
            
        wave_kx.append(kx_wbm)
        wave_ky.append(ky_wbm)
        wave_kz.append(kz_wbm)
        
        # Define Phi_w symbolically in NGSolve
        phi_w = exp(1j * (kx_wbm * x + ky_wbm * y + kz_wbm * (z - L)))
        wave_functions.append(phi_w)

In [54]:
# Assemble WBM Matrices
print(f"Assembling WBM Coupling Matrices for {total_waves} Bloch-Floquet waves...")
Z_wbm = np.zeros((total_waves, total_waves), dtype=complex)
Z_hyb = np.zeros((fes.ndof, total_waves), dtype=complex)
n_wb = CF((0, 0, -1))
for i in range(total_waves):
    dphi_i_dz = 1j*wave_kz[i]*wave_functions[i]
    cwf = LinearForm(fes)
    cwf += - dphi_i_dz * v * ds("top")
    with TaskManager():
        cwf.Assemble()
    Z_hyb[:, i] = cwf.vec.FV().NumPy()
    for j in range(total_waves):
        phi_j = wave_functions[j]
        integrand = Conj(dphi_i_dz) * phi_j
        
        # Integrate symbolically over the interface boundary
        val = Integrate(integrand, mesh, definedon=mesh.Boundaries("top"))
        Z_wbm[i, j] = val

print("shape of Z_wbm:", Z_wbm.shape)
print("shape of Z_hyb:", Z_hyb.shape)
print("condition number of Z_wbm:", np.linalg.cond(Z_wbm))

Assembling WBM Coupling Matrices for 1 Bloch-Floquet waves...
shape of Z_wbm: (1, 1)
shape of Z_hyb: (1746, 1)
condition number of Z_wbm: 1.0


In [55]:
# Hybrid model

# Convert sparse Z_fem to SciPy CSC format
row, col, val = Z_fem.mat.COO()
Z_fem_scipy = sp.coo_matrix((val, (row, col)), shape=(fes.ndof, fes.ndof)).tocsc()
Z_fem_free = Z_fem_scipy[free_indices, :][:, free_indices]
Z_hyb_free = Z_hyb[free_indices, :]

f_f_np = s_fem.vec.FV().NumPy()
s_fem_free = f_f_np[free_indices]

# Block Matrix 
top_row = sp.hstack([Z_fem_free, Z_hyb_free])
bottom_row = np.hstack([Z_hyb_free.conj().T, Z_wbm])
Global_Matrix = sp.vstack([top_row, bottom_row]).tocsc()

# Global RHS Vector 
s_wbm = np.zeros(total_waves, dtype=complex)
Global_RHS = np.concatenate([s_fem_free, s_wbm])

print(f"Is Global_Matrix sparse? {sp.issparse(Global_Matrix)}")
print(f"Exact type of Global_Matrix: {type(Global_Matrix)}")
print("condition number of Global_Matrix:", np.linalg.cond(Global_Matrix.toarray()))

Is Global_Matrix sparse? True
Exact type of Global_Matrix: <class 'scipy.sparse._csc.csc_matrix'>
condition number of Global_Matrix: 37493.34135123856


In [56]:
# Solve the dense/sparse hybrid system using SuperLU
solution = spla.spsolve(Global_Matrix, Global_RHS)
# =====================================================================
# 7. Post-Processing & Visualization
# =====================================================================
p_fem_vals = solution[:len(free_indices)]
p_wbm_factors = solution[len(free_indices):]

# Map back to an NGSolve GridFunction
p_fem_gf = GridFunction(fes)
p_fem_gf.vec.FV().NumPy()[free_indices] = p_fem_vals

In [57]:
# 1. Reconstruct Field
phase = exp(1j * (kx * x + ky * y))
p_true = p_fem_gf * phase

# 3. Visualize
print("Rendering total pressure field (PML zeroed out)...")
Draw(p_true, mesh, "Acoustic Pressure", animate_complex=True)

Rendering total pressure field (PML zeroed out)...


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Complex': {'phase': 0.0, 's…

BaseWebGuiScene